# SDH exp_013 — standalone exp12 pipeline audit
외부 실험 모듈을 사용하지 않는 독립 구현의 재현성과 누수 안전성을 한 단계씩 확인한다.

In [ ]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.exceptions import ConvergenceWarning

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'experiments').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'experiments').exists():
    raise RuntimeError('저장소 내부에서 노트북을 실행해 주세요.')
EXP_DIR = PROJECT_ROOT / 'experiments' / 'SDH' / 'exp_013_standalone_pipeline_audit'
RESULTS_DIR = EXP_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
if str(EXP_DIR) not in sys.path:
    sys.path.insert(0, str(EXP_DIR))

import standalone_pipeline as pipe
print('project root:', PROJECT_ROOT)

In [ ]:
data_dir = PROJECT_ROOT / 'data' / 'raw'
train = pd.read_csv(data_dir / 'train.csv')
test = pd.read_csv(data_dir / 'test.csv')
sample_submission = pd.read_csv(data_dir / 'sample_submission.csv')
genes = [column for column in train if column not in ('ID', 'SUBCLASS')]
labels = train['SUBCLASS'].reset_index(drop=True)
assert list(test.columns) == ['ID', *genes]
print('train:', train.shape, 'test:', test.shape, 'genes:', len(genes))

## 1. train-only vocabulary 불변성 검사
test의 순서·행 수·test-only token을 바꿔도 train vocabulary와 train 피처가 변하지 않아야 한다.

In [ ]:
audit_train = train[genes].iloc[:300].reset_index(drop=True)
audit_test_a = test[genes].iloc[:80].reset_index(drop=True)
audit_test_b = audit_test_a.iloc[::-1].reset_index(drop=True).copy()
audit_test_b.loc[0, genes[0]] = 'TEST_ONLY_A999Z'

train_a, test_a, vocab_a = pipe.fit_transform_pair(audit_train, audit_test_a, genes)
train_b, test_b, vocab_b = pipe.fit_transform_pair(audit_train, audit_test_b, genes)
assert vocab_a == vocab_b
assert (train_a.mutation != train_b.mutation).nnz == 0
assert (train_a.exact != train_b.exact).nnz == 0
assert (train_a.gene_type != train_b.gene_type).nnz == 0
assert 'TEST_ONLY_A999Z' not in vocab_b.exact_events
print('PASS: test 변경이 train vocabulary/행렬에 영향을 주지 않음')

## 2. standalone seed 42 CV
각 outer fold에서 train 분할로 vocabulary를 새로 fit하고 validation에는 적용만 한다. 기존 exp12 seed42 기준은 0.5291849039다.

In [ ]:
seed42_result = pipe.evaluate_seed(train, genes, seed=42, use_fixed_contrast=True)
display(pd.DataFrame([{key: value for key, value in seed42_result.items() if key != 'prediction'}]))
print('legacy exp12:', 0.5291849038818869)
print('delta:', seed42_result['oof_f1_macro'] - 0.5291849038818869)

## 3. 고정 contrast 제거 ablation
이 셀은 parity 검사가 통과한 뒤 실행한다. 고정 암종 쌍을 제거한 규칙 보수형 점수를 확인한다.

In [ ]:
no_contrast_result = pipe.evaluate_seed(train, genes, seed=42, use_fixed_contrast=False)
comparison = pd.DataFrame([
    {'case': 'standalone_fixed_contrast', 'f1_macro': seed42_result['oof_f1_macro']},
    {'case': 'standalone_no_contrast', 'f1_macro': no_contrast_result['oof_f1_macro']},
])
comparison['delta_vs_fixed'] = comparison['f1_macro'] - seed42_result['oof_f1_macro']
display(comparison)

## 4. full-train 제출 행렬 및 예측
원본 train/test는 결합하지 않는다. 출력 파일은 results/에 생성되며 git에서 제외된다.

In [ ]:
x_train, x_test, feature_names, submission_audit = pipe.build_design_matrices(
    train[genes], test[genes], labels, genes, seed=42, use_fixed_contrast=True
)
assert submission_audit['raw_train_test_concat'] is False
assert submission_audit['vocabulary_source'] == 'fit_frame_only'
assert x_train.shape[1] == x_test.shape[1] == len(feature_names)
display(pd.DataFrame([submission_audit]))

In [ ]:
model = pipe.make_model(seed=42)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always', ConvergenceWarning)
    model.fit(x_train, labels)
prediction = model.predict(x_test)
warning_count = sum(issubclass(item.category, ConvergenceWarning) for item in caught)
assert warning_count == 0
submission = sample_submission.copy()
submission['SUBCLASS'] = prediction
submission_path = RESULTS_DIR / 'submission_exp013_standalone_fixed_contrast_seed42.csv'
submission.to_csv(submission_path, index=False)
print('saved:', submission_path)

## 5. 기존 exp12 제출 예측 완전 동일성
기존 파일이 로컬 results에 있을 때 2,546행 전체를 비교한다.

In [ ]:
legacy_path = PROJECT_ROOT / 'experiments' / 'SDH' / 'exp_012_enrichment_stability' / 'results' / 'submission_exp012_b04_gene_type_shrink10_seed42.csv'
if legacy_path.exists():
    legacy_submission = pd.read_csv(legacy_path)
    assert legacy_submission['ID'].equals(submission['ID'])
    changed = int((legacy_submission['SUBCLASS'] != submission['SUBCLASS']).sum())
    print('changed rows vs exp12:', changed)
    assert changed == 0, 'standalone 제출 예측이 기존 exp12와 다릅니다.'
else:
    print('legacy submission 파일이 없어 예측 parity를 건너뜀:', legacy_path)

## 6. 3-seed 확인
seed42 parity를 먼저 확인한 뒤 52, 62를 실행한다. 기존 exp12 평균 기준은 0.5282357113이다.

In [ ]:
seed_results = [seed42_result]
for seed in (52, 62):
    seed_results.append(pipe.evaluate_seed(train, genes, seed=seed, use_fixed_contrast=True))
per_seed = pd.DataFrame([{key: value for key, value in result.items() if key not in ('prediction', 'fold_scores')} for result in seed_results])
display(per_seed)
print('3-seed mean:', per_seed['oof_f1_macro'].mean())
print('legacy mean:', 0.5282357119657942)
per_seed.to_csv(RESULTS_DIR / 'per_seed_standalone.csv', index=False)